# atml-pa3-t1-train-fullft

Stage: `train:full_ft`

Self-contained ATML PA3 Task 1 runner. Source is embedded in the notebook (and also attached as a dataset when available). Artifacts go to `/kaggle/working/results`.


In [ ]:
import os, sys, shutil, pathlib, zipfile, io, base64
STAGE = 'train:full_ft'
SMOKE_STEPS = int(os.environ.get('ATML_SMOKE_STEPS', '4'))
dst = pathlib.Path('/kaggle/working/task1')
if dst.exists():
    shutil.rmtree(dst)
dst.mkdir(parents=True)

# 1) Prefer attached dataset if present
inp = pathlib.Path('/kaggle/input')
src = None
candidates = []
if inp.exists():
    candidates += sorted(inp.glob('*'))
    candidates += sorted(inp.glob('datasets/*/*'))
    candidates += sorted(inp.glob('*/*'))
for p in candidates:
    if p.is_dir() and (p/'src').is_dir() and (p/'configs').is_dir():
        src = p
        break
if src is not None:
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print('Using attached dataset', src)
else:
    print('No usable attached dataset; extracting embedded code zip')
    CODE_B64 = '''...'''
    raw = base64.b64decode(CODE_B64)
    with zipfile.ZipFile(io.BytesIO(raw)) as zf:
        zf.extractall(dst)
assert (dst/'src').is_dir() and (dst/'configs').is_dir(), list(dst.iterdir())
os.chdir(dst)
sys.path.insert(0, str(dst))
print('CODE_ROOT', dst)
print('STAGE', STAGE)


In [ ]:
import torch, subprocess, sys
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), 'VRAM_GiB', round(torch.cuda.get_device_properties(0).total_memory/2**30, 2))
pkgs = [
    'datasets', 'peft', 'accelerate', 'sentencepiece', 'pyyaml',
    'lm-eval', 'einops', 'safetensors', 'matplotlib', 'seaborn', 'pandas', 'tabulate',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'transformers>=4.51.0'] + pkgs)


In [ ]:
import os, subprocess, sys, pathlib, json, time, traceback, shutil
os.environ['ATML_ARTIFACTS_ROOT'] = '/kaggle/working/results'
os.environ['HF_HOME'] = '/tmp/hf_home'
os.environ['HF_HUB_CACHE'] = '/tmp/hf_home'
os.environ['TRANSFORMERS_CACHE'] = '/tmp/hf_home'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
pathlib.Path('/kaggle/working/results').mkdir(parents=True, exist_ok=True)
t0 = time.time()
cmd = [sys.executable, '-m', 'src.run_stage', '--stage', STAGE, '--artifacts-root', '/kaggle/working/results']
if STAGE == 'smoke':
    cmd += ['--smoke-steps', str(SMOKE_STEPS)]
print('Running', cmd, flush=True)
try:
    rc = subprocess.call(cmd)
except Exception:
    pathlib.Path('/kaggle/working/results/failure.txt').write_text(traceback.format_exc())
    raise
print('RC', rc, 'elapsed_min', round((time.time()-t0)/60, 2))
status = {'stage': STAGE, 'returncode': rc, 'elapsed_sec': time.time()-t0}
pathlib.Path('/kaggle/working/results/stage_status.json').write_text(json.dumps(status, indent=2))
# Keep working directory small for output downloads
shutil.rmtree('/kaggle/working/task1', ignore_errors=True)
shutil.rmtree('/kaggle/working/hf_home', ignore_errors=True)
if rc != 0:
    # Capture tail of process by re-running is impossible; rely on logs + stage_status
    raise SystemExit(rc)


In [ ]:
import pathlib
root = pathlib.Path('/kaggle/working/results')
files = sorted(str(p.relative_to(root)) for p in root.rglob('*') if p.is_file())
print('artifact_files', len(files))
for p in files[:120]:
    print(p)
status = (root/'stage_status.json')
print(status.read_text() if status.exists() else 'no status')
